# SI_robustness_gpt_validation.ipynb

**Purpose:** Use an LLM to judge whether each candidate name match in the
validation sample is a genuine self-promotion match, correct the LLM's errors
by manual review, and use the resulting labels as ground truth to compute
accuracy, precision, recall and F1 for every pair of similarity thresholds
-> **Fig. S8 input**.

**Manuscript:** Gender differences in online visibility of early-career researchers

**Input:** `data/5_name_match_validation_sample.csv` -- 100 borderline name
matches (similarity score 0.5--0.9; 70 classified as no match and 30 as a match
by the algorithm at the paper's thresholds). Columns used:
`first_name`, `last_name` (author name in Scopus), `most_likely_name`
(display name / handle of the most similar tweeter), `max_score`
(name-similarity score, 0--1), `chinese_name` (1 if the name is classified as
Chinese, stricter threshold), and `self_pro` (the algorithm's decision at the
paper's thresholds, 0.6 / 0.8).

The file contains real author and Twitter names (needed for the LLM
judgement) and must **not** be published.

**Required environment variable:** `GWDG_API_KEY`.

**Output** (in `result/`):
- `5_name_match_validation_sample_testresult.csv` -- **Fig. S8 input**
  (`SI_robustness_S8_S9.R` writes it to `tables/fig_s8.csv`)


In [ ]:
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

DATA_DIR = Path("data")
RESULT_DIR = Path("result")
RESULT_DIR.mkdir(parents=True, exist_ok=True)

INPUT_FILE = DATA_DIR / "5_name_match_validation_sample.csv"
METRICS_OUT = RESULT_DIR / "5_name_match_validation_sample_testresult.csv"

### 1. Load the validation sample

`case_id` is the row position in the file (0--99). The manual overrides in
Section 3 refer to these `case_id`s, so do **not** re-sort or filter the file
before this step.

In [ ]:
sample = pd.read_csv(INPUT_FILE)
sample["case_id"] = np.arange(len(sample))
sample = sample.set_index("case_id")[[
    "first_name", "last_name", "most_likely_name", "max_score", "chinese_name", "self_pro",
]]

# One case (case_id 17) has no name or similarity data. It is a true negative
# at every threshold, so dropping it does not change precision, recall or F1
# (only accuracy, marginally).
incomplete = sample[sample["max_score"].isna()]
if len(incomplete):
    print(f"Dropping {len(incomplete)} incomplete case(s): case_id {list(incomplete.index)}")
sample = sample.dropna(subset=["max_score"])
print(f"{len(sample)} cases; algorithm matches: {int(sample['self_pro'].sum())}")

### 2. LLM judgement

Set the API key before launching Jupyter, e.g. `export GWDG_API_KEY=your_key_here` -- **never hardcode API keys
in a notebook that will be made public.**

In [ ]:
PROMPT = """You are given a person's first name, last name, and display name.

First name: {first}
Last name: {last}
Display name: {display}

Determine whether the display name refers to the same person (i.e., if it
contains, abbreviates, or combines the first and last name).
Answer with one word: "Yes" or "No"."""


def ask_llm(client, model, first, last, display, max_retries=5):
    """Return 1 (Yes), 0 (No) or None if the request failed."""
    prompt = PROMPT.format(first=first, last=last, display=display)
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                temperature=0,
            )
            answer = response.choices[0].message.content.strip().lower()
            return 1 if answer.startswith("yes") else 0 if answer.startswith("no") else None
        except Exception as e:
            if "429" in str(e):  # rate limit
                wait = 2 ** attempt + np.random.random()
                print(f"Rate limit hit. Waiting {wait:.1f} s...")
                time.sleep(wait)
            else:
                print(f"Error: {e}")
                return None
    return None


from openai import OpenAI

api_key = os.environ.get("GWDG_API_KEY")
if api_key is None:
    raise EnvironmentError("Set the GWDG_API_KEY environment variable before running.")
client = OpenAI(api_key=api_key, base_url="https://chat-ai.academiccloud.de/v1")
model = "openai-gpt-oss-120b"

labels = {}
for case_id, row in sample.iterrows():
    labels[case_id] = ask_llm(client, model, row["first_name"],
                              row["last_name"], row["most_likely_name"])
    time.sleep(0.5)

sample["gpt_match_test"] = pd.Series(labels).reindex(sample.index)
if sample["gpt_match_test"].isna().any():
    print("Warning: LLM label missing for case_id", list(sample.index[sample["gpt_match_test"].isna()]))

### 3. Manual review of LLM/algorithm disagreements

Cases where the LLM and the algorithm disagree were reviewed by hand. The
cases listed in `manual_overrides` were confirmed (by inspecting the names and
handles) to be genuine matches that the LLM had labelled as non-matches; they
are set to 1 in `gpt_match_test`, which then serves as the ground truth.

In [ ]:
disagreements = sample[sample["gpt_match_test"] != sample["self_pro"]]
print(f"{len(disagreements)} disagreements between the LLM and the algorithm")
disagreements[["first_name", "last_name", "most_likely_name", "max_score", "gpt_match_test", "self_pro"]]

In [ ]:
manual_overrides = [72, 77, 80, 81, 82, 89, 95, 99]  # case_ids confirmed as genuine matches

missing = [c for c in manual_overrides if c not in sample.index]
if missing:
    raise ValueError(f"case_id(s) {missing} not in the sample -- check the input file order.")

sample.loc[manual_overrides, "gpt_match_test"] = 1
sample["gpt_match_test"] = sample["gpt_match_test"].astype(int)
print(f"Genuine matches after manual review: {sample['gpt_match_test'].sum()} of {len(sample)}")

### 4. Threshold sweep against the validated labels -> Fig. S8

Same matching rule as in the main analysis: a match if the similarity score
exceeds the general threshold, and for Chinese names only if it also reaches
the Chinese-name threshold.

In [ ]:
def predict_match(df, general_threshold, chinese_threshold):
    is_match = df["max_score"] > general_threshold
    is_match &= ~((df["chinese_name"] == 1) & (df["max_score"] < chinese_threshold))
    return is_match.astype(int)


rows = []
for general_t in np.round(np.arange(0.5, 1.01, 0.1), 2):
    for chinese_t in np.round(np.arange(0.5, 1.01, 0.1), 2):
        y_true = sample["gpt_match_test"]
        y_pred = predict_match(sample, general_t, chinese_t)
        rows.append({
            "general_threshold": general_t,
            "chinese_threshold": chinese_t,
            "accuracy": accuracy_score(y_true, y_pred),
            "precision": precision_score(y_true, y_pred, zero_division=0),
            "recall": recall_score(y_true, y_pred, zero_division=0),
            "f1": f1_score(y_true, y_pred, zero_division=0),
        })

threshold_result = pd.DataFrame(rows)
threshold_result.to_csv(METRICS_OUT, index=False)
print(f"Wrote {METRICS_OUT} ({len(threshold_result)} threshold pairs)")

best = threshold_result.loc[threshold_result["f1"].idxmax()]
print(f"Best F1: {best['f1']:.3f} at general = {best['general_threshold']}, Chinese = {best['chinese_threshold']}")